# OUTCOME SEVERITY: baseline age, sex, genetics
import functions from `cardi_utils.py`.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from cardi_utils import (
    load_data,
    get_snp_cols,
    severity_weights,
    make_stratified_folds,
    train_one_fold_lgbm,
    sample_params_binary,
    weighted_logloss_binary,
    weighted_dummy_logloss_binary,
    weighted_worst_logloss_binary,
    rescaled_weighted_logloss,
    rank_snps_by_shap_severity_lgbm,
)

pd.set_option("display.max_columns", None)

train, test = load_data()
snp_cols = get_snp_cols(train)

## Features

In [2]:
fixed_cols_severity = ["Age_Baseline", "Genre"]

X = train[fixed_cols_severity + snp_cols].copy()
y = train["OUTCOME SEVERITY"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Model

In [3]:
def fit_lgbm_severity_rescaled_logloss_cv(
    train: pd.DataFrame,
    target_col: str,
    base_cols: list[str],
    snp_cols: list[str],
    stability_df: pd.DataFrame | None = None,
    selected_snp_top_n: int = 20,
    candidate_pool_size: int | None = None,
    use_pca: bool = True,
    n_pcs: int = 10,
    add_snp_sum: bool = True,
    test_size: float = 0.2,
    split_seed: int = 42,
    severity_label_weights: dict[int, float] | None = None,
    use_sample_weights: bool = True,  # affects TRAINING ONLY
    n_splits_cv: int = 5,
    n_param_samples: int = 40,
    num_boost_round: int = 4000,
    early_stopping_rounds: int = 100,
    verbose_cv: bool = False,
):

    if severity_label_weights is None:
        severity_label_weights = {0: 1.5, 1: 1.0}

    y_all = train[target_col].astype(int)
    tr_idx, te_idx = train_test_split(
        train.index, test_size=test_size, stratify=y_all, random_state=split_seed
    )
    y_tr = y_all.loc[tr_idx].values
    y_te = y_all.loc[te_idx].values

    # SNP ranking 
    snp_cols = [c for c in snp_cols if c in train.columns]
    if not snp_cols:
        raise ValueError("No SNP columns found in train.")

    if stability_df is not None and "snp" in stability_df.columns:
        ranked = [s for s in stability_df["snp"] if s in snp_cols]
        if not ranked:
            ranked = snp_cols.copy()
    else:
        ranked = snp_cols.copy()

    selected = ranked[:selected_snp_top_n]
    remaining = [s for s in ranked if s not in selected]
    candidate_pool = remaining if candidate_pool_size is None else remaining[:candidate_pool_size]

    if use_pca and len(candidate_pool) < max(n_pcs, 2):
        raise ValueError("Not enough SNPs in candidate_pool for PCA.")

    fixed_cols = base_cols + selected

    X_fixed_tr = train.loc[tr_idx, fixed_cols].copy()
    X_fixed_te = train.loc[te_idx, fixed_cols].copy()

    X_snp_tr = train.loc[tr_idx, candidate_pool].copy()
    X_snp_te = train.loc[te_idx, candidate_pool].copy()

    X_tr_df = X_fixed_tr.copy()
    X_te_df = X_fixed_te.copy()

    pca = None
    pc_cols: list[str] = []

    if use_pca:
        pca = PCA(n_components=n_pcs, random_state=split_seed)
        pcs_tr = pca.fit_transform(X_snp_tr.values)
        pcs_te = pca.transform(X_snp_te.values)
        pc_cols = [f"SNP_PC{i+1}" for i in range(n_pcs)]
        X_tr_df = pd.concat([X_tr_df, pd.DataFrame(pcs_tr, index=tr_idx, columns=pc_cols)], axis=1)
        X_te_df = pd.concat([X_te_df, pd.DataFrame(pcs_te, index=te_idx, columns=pc_cols)], axis=1)

    if add_snp_sum:
        # SNP sum is computed over the candidate_pool (matches original intent)
        X_tr_df["SNP_SUM"] = X_snp_tr.sum(axis=1).values
        X_te_df["SNP_SUM"] = X_snp_te.sum(axis=1).values

    feature_cols = list(X_tr_df.columns)


    # Evaluation weights (used for metrics)
    w_tr_eval = np.array([severity_label_weights[int(y)] for y in y_tr], dtype=float)
    w_te_eval = np.array([severity_label_weights[int(y)] for y in y_te], dtype=float)

    # Training weights
    w_tr_fit = w_tr_eval if use_sample_weights else None
    w_te_fit = w_te_eval if use_sample_weights else None

    # Baselines (computed on TRAIN distribution only; used to rescale fold scores)
    dummy_ll_tr = weighted_dummy_logloss_binary(y_tr, severity_label_weights)
    worst_ll_tr = weighted_worst_logloss_binary(y_tr, severity_label_weights)

    folds = make_stratified_folds(y_tr, n_splits=n_splits_cv, seed=split_seed)
    rng = np.random.default_rng(split_seed)

    best_score = -np.inf
    best_params: dict | None = None
    best_num_boost: int | None = None

    X_tr_np = X_tr_df.values

    for _ in range(n_param_samples):
        params = sample_params_binary(rng=rng)

        fold_scores: list[float] = []
        fold_best_iters: list[int] = []

        for fold in folds:
            tr_i, va_i = fold.train_idx, fold.valid_idx

            booster, it, p_va = train_one_fold_lgbm(
                params=params,
                X_tr=X_tr_np[tr_i],
                y_tr=y_tr[tr_i],
                X_va=X_tr_np[va_i],
                y_va=y_tr[va_i],
                feature_cols=feature_cols,
                num_boost_round=num_boost_round,
                early_stopping_rounds=early_stopping_rounds,
                verbose=verbose_cv,
                w_tr=None if w_tr_fit is None else w_tr_fit[tr_i],
                w_va=None if w_tr_fit is None else w_tr_fit[va_i],
            )

            # Fold metric: rescaled weighted logloss (higher is better)
            wll = weighted_logloss_binary(y_tr[va_i], p_va, severity_label_weights)
            rll = rescaled_weighted_logloss(wll, dummy_ll_tr, worst_ll_tr)

            fold_scores.append(float(rll))
            fold_best_iters.append(int(it))

        mean_rll = float(np.mean(fold_scores))
        mean_it = int(np.round(np.mean(fold_best_iters)))

        if mean_rll > best_score:
            best_score = mean_rll
            best_params = params
            best_num_boost = max(1, mean_it)

    if best_params is None or best_num_boost is None:
        raise RuntimeError("Severity CV search failed.")

# final train

    dtrain_full = lgb.Dataset(
        X_tr_df.values, label=y_tr, weight=w_tr_fit, feature_name=feature_cols, free_raw_data=True
    )
    dtest = lgb.Dataset(
        X_te_df.values, label=y_te, weight=w_te_fit, feature_name=feature_cols, reference=dtrain_full, free_raw_data=True
    )

    final_model = lgb.train(
        params=best_params,
        train_set=dtrain_full,
        num_boost_round=best_num_boost,
        valid_sets=[dtest],
        valid_names=["test"],
        callbacks=[
            lgb.early_stopping(early_stopping_rounds, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    best_it_final = int(final_model.best_iteration or best_num_boost)


    # Final evaluation

    p_tr = final_model.predict(X_tr_df.values, num_iteration=best_it_final)
    p_te = final_model.predict(X_te_df.values, num_iteration=best_it_final)

    wll_tr = weighted_logloss_binary(y_tr, p_tr, severity_label_weights)
    wll_te = weighted_logloss_binary(y_te, p_te, severity_label_weights)

    rll_tr = rescaled_weighted_logloss(wll_tr, dummy_ll_tr, worst_ll_tr)
    rll_te = rescaled_weighted_logloss(wll_te, dummy_ll_tr, worst_ll_tr)

    metrics = {
        "cv_best_rescaled_wll": float(best_score),
        "train_rescaled_wll": float(rll_tr),
        "test_rescaled_wll": float(rll_te),
        "best_num_boost_round": int(best_num_boost),
        "best_iteration_final": int(best_it_final),
        "used_sample_weights_for_training": bool(use_sample_weights),
    }

    return best_params, final_model, pca, metrics, candidate_pool, feature_cols, selected

## Rank SNPs by weighted mean |SHAP|

In [4]:
sev_rank_df = rank_snps_by_shap_severity_lgbm(
    train=train,
    target_col="OUTCOME SEVERITY",
    base_cols=fixed_cols_severity,
    snp_cols=snp_cols,
    use_weights_for_training=True,
    use_weights_for_shap=True,
)

In [5]:
sev_rank_df

,snp,mean_abs_shap,rank
0,SNP21,0.119570,1
1,SNP282,0.082033,2
2,SNP10,0.070791,3
3,SNP99,0.051411,4
4,SNP175,0.039460,5
...,...,...,...
283,SNP28,0.000000,284
284,SNP38,0.000000,285
285,SNP30,0.000000,286
286,SNP31,0.000000,287


## Fit

In [6]:
best_params_sev, model_sev, pca_sev, metrics_sev, pool_sev, feat_sev, trusted_sev = fit_lgbm_severity_rescaled_logloss_cv(
    train=train,
    target_col="OUTCOME SEVERITY",
    candidate_pool_size=80,
    base_cols=fixed_cols_severity,
    snp_cols=snp_cols,
    stability_df=sev_rank_df,
    severity_label_weights=severity_weights,
    use_sample_weights=True,
)
print(metrics_sev)

{'cv_best_rescaled_wll': 0.1313118020648695, 'train_rescaled_wll': 0.09415015860068765, 'test_rescaled_wll': 0.04662202644961766, 'best_num_boost_round': 200, 'best_iteration_final': 9, 'used_sample_weights_for_training': True}


In [7]:
#model_sev.save_model('severity_model_sub5.txt')

## Predict on test

In [8]:
ID_COL = "trustii_id"
N_PCS = 10
pc_cols = [f"SNP_PC{i+1}" for i in range(N_PCS)]

# Base + trusted raw SNPs + remaining SNPs (for PCA/SNP_sum)
X_test = test[fixed_cols_severity + trusted_sev + pool_sev].copy()

# PCA transform 
if pca_sev is not None:
    pcs = pca_sev.transform(X_test[pool_sev].to_numpy())
    for i, c in enumerate(pc_cols):
        X_test[c] = pcs[:, i]

X_test["SNP_SUM"] = X_test[pool_sev].sum(axis=1)

X_test.drop(columns=pool_sev, inplace=True)
X_test = X_test[feat_sev]

proba_1 = model_sev.predict(
    X_test.values,
    num_iteration=getattr(model_sev, "best_iteration", None)
)

pred_df_severity = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "OUTCOME SEVERITY": proba_1, 
})
pred_df_severity

,trustii_id,OUTCOME SEVERITY
0,1,0.196824
1,2,0.290172
2,3,0.247374
3,4,0.236243
4,5,0.270367
...,...,...
144,145,0.266493
145,146,0.196824
146,147,0.303974
147,148,0.244773


In [9]:
pred_df_severity.to_csv("pred_severity.csv", index=False)